# 03 — Optimización Económica de Políticas de Crédito y P&L
**Credit Policy Optimizer** — Modelado Financiero de Decisiones en Originación

### Objetivos del Experimento:
1. Demostrar la falacia de evaluar modelos crediticios con cortes ingenuos ($p=0.5$ o $F_1$-max).
2. Construir curvas de política financiera: **Tasa de Aprobación vs. Tasa de Mora vs. P&L Neto Esperado vs. ROE**.
3. Encontrar el umbral óptimo global $p^*$ y compararlo con el umbral breakeven teórico.
4. Aplicar **Optimización con Restricciones de Negocio** (ej. tope de mora de cartera al 3.5%).
5. Evaluar la asignación de límites ajustados por riesgo (*Risk-based limit sizing*).


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from credit_policy_optimizer.data.generator import PortfolioSimulator
from credit_policy_optimizer.models.pipeline import (
    train_credit_pipeline,
    DEFAULT_NUMERIC_FEATURES,
    DEFAULT_CATEGORICAL_FEATURES
)
from credit_policy_optimizer.models.calibration import calibrate_pipeline
from credit_policy_optimizer.decision.economics import (
    CreditPolicyOptimizer,
    EconomicParameters,
    compute_breakeven_pd
)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)


## 1. Generación de Cartera y Obtención de Probabilidades Calibradas


In [ ]:
sim = PortfolioSimulator(seed=42)
portfolio = sim.simulate(n_samples=20_000)

features = DEFAULT_NUMERIC_FEATURES + DEFAULT_CATEGORICAL_FEATURES
X = portfolio.select(features)
y = portfolio["default_flag"].to_numpy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

raw_model = train_credit_pipeline(X_train, y_train)
calibrated_model = calibrate_pipeline(raw_model, X_val, y_val, method="isotonic")

# Inyectar probabilidades predichas al portafolio de evaluación
preds_pd = calibrated_model.predict_proba(portfolio)[:, 1]
eval_portfolio = portfolio.with_columns(pl.Series("pd", preds_pd))

eval_portfolio.select(["monthly_income", "debt_to_income", "loan_amount", "pd", "default_flag"]).head(5)


## 2. Parámetros Financieros y Umbral Breakeven Teórico
Definimos los parámetros económicos institucionales:
- **Tasa Activa Nominal ($r$)**: 18% anual.
- **Costo de Fondos ($c$)**: 5% anual.
- **Loss Given Default (LGD)**: 45%.
- **Plazo ($T$)**: 12 meses.


In [ ]:
params = EconomicParameters(
    default_lgd=0.45,
    default_cost_of_funds=0.05,
    default_interest_rate=0.18,
    default_loan_term_months=12
)

optimizer = CreditPolicyOptimizer(eval_portfolio, params=params)

p_star_theoretical = compute_breakeven_pd(
    interest_rate=params.default_interest_rate,
    cost_of_funds=params.default_cost_of_funds,
    lgd=params.default_lgd,
    term_months=params.default_loan_term_months
)

print(f"=== Parámetros Financieros ===")
print(f"Margen Neto Activo (r - c):       {(params.default_interest_rate - params.default_cost_of_funds)*100:.1f}%")
print(f"Costo Total de Quiebra (LGD + c): {(params.default_lgd + params.default_cost_of_funds)*100:.1f}%")
print(f"Umbral Breakeven Teórico (p*):    {p_star_theoretical * 100:.2f}%")


## 3. Simulación de la Curva de Trade-off (Barrido de Umbrales)
Exploramos cómo varían la aprobación, la mora, la exposición y el P&L neto según el punto de corte $p$.


In [ ]:
tradeoff = optimizer.compute_tradeoff_curve(num_points=120)
tradeoff_pd = tradeoff.to_pandas()

tradeoff.head(5)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: P&L Neto Esperado vs. Umbral
axes[0].plot(tradeoff_pd["threshold"], tradeoff_pd["net_financial_margin"] / 1000, label="P&L Neto Esperado ($k)", color="#2b5c8f", lw=2.5)
opt_result = optimizer.optimize_threshold()
best_t = opt_result.optimal_threshold
best_pnl = opt_result.optimal_policy.expected_pnl / 1000

axes[0].axvline(best_t, color="green", linestyle="--", label=f"Umbral Óptimo p* = {best_t:.3f}")
axes[0].axvline(0.50, color="crimson", linestyle=":", label="Corte Ingenuo ML (p = 0.50)")
axes[0].axhline(0, color="black", lw=1)
axes[0].set_title("Función de Utilidad Financiera (P&L Neto vs. Umbral de Decisión)")
axes[0].set_xlabel("Umbral de Probabilidad de Aprobación (p)")
axes[0].set_ylabel("P&L Neto Esperado (Miles de USD)")
axes[0].legend()

# Gráfico 2: Tasa de Aprobación vs. Tasa de Mora de Cartera Aprobada
axes[1].plot(tradeoff_pd["threshold"], tradeoff_pd["approval_rate"] * 100, label="Tasa de Aprobación (%)", color="#7570b3", lw=2)
axes[1].plot(tradeoff_pd["threshold"], tradeoff_pd["expected_default_rate"] * 100, label="Tasa de Mora Aprobados (%)", color="#d95f02", lw=2)
axes[1].axvline(best_t, color="green", linestyle="--")
axes[1].set_title("Dinámica de Concesión vs. Calidad de Cartera")
axes[1].set_xlabel("Umbral de Probabilidad (p)")
axes[1].set_ylabel("Porcentaje (%)")
axes[1].legend()

plt.tight_layout()
plt.show()


## 4. Comparativa de Resultados: Ingenuo vs. Óptimo Financiero


In [ ]:
baseline_eval = optimizer.evaluate_policy(0.50)
optimal_eval = opt_result.optimal_policy

print("=" * 60)
print(f"POLÍTICA INGENUA ML (p = 0.50):")
print(f"  - Tasa de Aprobación: {baseline_eval.approval_rate*100:.1f}% ({baseline_eval.approved_count:,} créditos)")
print(f"  - Tasa de Mora Esperada: {baseline_eval.expected_default_rate*100:.2f}%")
print(f"  - Pérdida Esperada:   ${baseline_eval.expected_loss:,.2f}")
print(f"  - P&L Neto Esperado:  ${baseline_eval.expected_pnl:,.2f}")
print("-" * 60)
print(f"POLÍTICA ÓPTIMA FINANCIERA (p = {best_t:.3f}):")
print(f"  - Tasa de Aprobación: {optimal_eval.approval_rate*100:.1f}% ({optimal_eval.approved_count:,} créditos)")
print(f"  - Tasa de Mora Esperada: {optimal_eval.expected_default_rate*100:.2f}%")
print(f"  - Pérdida Esperada:   ${optimal_eval.expected_loss:,.2f}")
print(f"  - P&L Neto Esperado:  ${optimal_eval.expected_pnl:,.2f}")
print(f"  - MEJORA NETA:        +${opt_result.incremental_pnl_vs_baseline:,.2f}")
print("=" * 60)


## 5. Optimización con Restricciones Operativas
En la práctica, la gerencia de riesgo impone límites de apetito de riesgo, por ejemplo:
- *Objetivo*: Maximizar P&L.
- *Restricción*: Tasa de mora promedio de cartera aprobada $\le 3.5\%$.
- *Restricción*: Tasa mínima de aprobación comercial $\ge 40\%$.

Probamos el método `optimize_constrained_policy`:


In [ ]:
constrained_policy = optimizer.optimize_constrained_policy(
    max_default_rate=0.035,
    min_approval_rate=0.40,
    metric="expected_pnl"
)

print(f"=== Política Optimizada con Restricciones ===")
print(f"Umbral Seleccionado:    {constrained_policy.threshold:.4f}")
print(f"Tasa de Aprobación:     {constrained_policy.approval_rate * 100:.2f}% (Restricción >= 40%)")
print(f"Tasa de Mora Cartera:   {constrained_policy.expected_default_rate * 100:.2f}% (Restricción <= 3.5%)")
print(f"P&L Neto Esperado:      ${constrained_policy.expected_pnl:,.2f}")
print(f"Retorno s/ Exposición:  {constrained_policy.return_on_exposure * 100:.2f}%")


## 6. Asignación de Límites de Crédito Ajustados por Riesgo
En lugar de una decisión binaria rígida (otorgar el 100% de lo solicitado o rechazar), evaluamos la política de dimensionamiento (*limit sizing*):
$$\text{Monto Otorgado}_i = \text{Monto Solicitado}_i \times \left(1 - \frac{PD_i}{p^*}\right)^{1.5}$$


In [ ]:
amount_res = optimizer.optimize_amount_policy(risk_sensitivity=1.5, min_approval_amount=500.0)

print("=== Política de Límites Ajustados por Riesgo ===")
print(f"Aprobación de Solicitudes:  {amount_res.approval_rate * 100:.1f}%")
print(f"Exposición Asignada:        ${amount_res.allocated_exposure:,.2f} (de ${amount_res.requested_exposure:,.2f} solicitados)")
print(f"P&L Neto Resultante:        ${amount_res.expected_pnl:,.2f}")
print(f"ROE sobre Cartera:          {amount_res.return_on_exposure * 100:.2f}%")


## 7. Conclusiones para el Comité de Crédito
1. **La regla $p=0.5$ destruye valor**: Aprobar con $p=0.5$ incurre en pérdidas masivas porque el costo de default supera ampliamente el margen de intereses.
2. **El óptimo económico $p^* \approx 0.20$** maximiza la ganancia institucional neta.
3. Las restricciones operativas permiten conciliar metas comerciales con el apetito de riesgo regulatorio.
